# IsyaratKu — Latih model abjad (`letters`)

Model abjad dilatih dari **dataset publik** (DECISIONS #21), bukan rekaman di aplikasi:

| Sumber | Isi | Lisensi | Peran |
| --- | --- | --- | --- |
| [Kaggle — Abjad Bahasa Isyarat Indonesia (BISINDO)](https://www.kaggle.com/datasets/meisyavira/abjad-bahasa-isyarat-indonesia-bisindo), Meisya Vira (gabungan dengan dataset Achmad Noer) | 26 huruf × 25 foto | ODbL | data utama: latih, validasi, uji |
| [Roboflow — BISINDO Dataset v3](https://universe.roboflow.com/bisindo/bisindo-dataset) | 22 huruf (tanpa N, P, R, T), grayscale 244×244 | CC BY 4.0 | opsional, hanya data latih (DECISIONS #17) |

Setiap foto dijalankan lewat MediaPipe HandLandmarker (model yang sama dengan aplikasi) → 21 titik per tangan →
**126 fitur** persis seperti `src/lib/features.ts` (DECISIONS #16, #19). Model: Dense 128 → Dropout 0,3 → Dense 64 → Softmax 26 (PRD §11.2).

Jalankan di **Google Colab**: Runtime → Run all. Hasil: `letters-model.zip` (salin isinya ke `public/models/letters/`)
dan `letters-report.zip` (akurasi, confusion matrix, kurva latih untuk laporan).

## 1. Instalasi

In [ ]:
# TF.js reads Keras 2 model configs; Colab's TensorFlow ships Keras 3, so use the legacy tf_keras.
# The tensorflowjs package is NOT used: it breaks on Python 3.13 / NumPy 2 (DECISIONS #23).
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
%pip install -q tf_keras scikit-learn matplotlib mediapipe kagglehub pyyaml

## 2. Label

Urutan label = `src/data/vocab.json` (indeks output model). Di Colab, unggah `vocab.json` saat diminta.

In [ ]:
import glob
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

vocab_path = Path("../src/data/vocab.json")
if not vocab_path.exists():
    vocab_path = DATA_DIR / "vocab.json"
    if not vocab_path.exists():
        print("Pilih file src/data/vocab.json")
        vocab_path.write_bytes(next(iter(files.upload().values())))
VOCAB = json.loads(vocab_path.read_text(encoding="utf-8"))
LETTERS = VOCAB["letters"]  # output index = position in this list; never reorder
assert len(LETTERS) == 26, LETTERS
print("Label:", " ".join(LETTERS))

## 3. Ekstraksi fitur

`normalize_hand` dan `to_features` adalah salinan `src/lib/features.ts`; kesamaannya dicek terhadap versi TypeScript
(selisih ≤ pembulatan 3 desimal). Kiri/kanan memakai label MediaPipe apa adanya, sama dengan `toHandFrame()` di
`src/lib/landmarks.ts` (DECISIONS #10).

In [ ]:
import math
import uuid
from collections import Counter
from datetime import datetime, timezone

import mediapipe as mp
import numpy as np
from PIL import Image
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

LANDMARKS = 21
PER_HAND = LANDMARKS * 3   # 63
FEATURES = PER_HAND * 2    # 126
DECIMALS = 3


def normalize_hand(points, aspect=1.0):
    """Python port of normalizeHand() in src/lib/features.ts. Keep both in sync."""
    wrist, ref = points[0], points[9]
    size = math.sqrt(((ref[0] - wrist[0]) * aspect) ** 2 + (ref[1] - wrist[1]) ** 2
                     + ((ref[2] - wrist[2]) * aspect) ** 2)
    if not size > 1e-6:
        return [0.0] * PER_HAND
    out = []
    for x, y, z in points:
        out += [(x - wrist[0]) * aspect / size, (y - wrist[1]) / size, (z - wrist[2]) * aspect / size]
    return out


def to_features(left, right, aspect=1.0):
    """126 features: left hand then right, missing hand = zeros (toFeatures in features.ts)."""
    feats = (normalize_hand(left, aspect) if left else [0.0] * PER_HAND) + \
            (normalize_hand(right, aspect) if right else [0.0] * PER_HAND)
    return [round(v, DECIMALS) for v in feats]


_demo = [(0.5 + 0.01 * i, 0.5 - 0.02 * i, 0.001 * i) for i in range(LANDMARKS)]
_f = normalize_hand(_demo, aspect=4 / 3)
assert _f[:3] == [0.0, 0.0, 0.0] and abs(math.hypot(*_f[27:30]) - 1) < 1e-9

import urllib.request
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/latest/hand_landmarker.task")
if not Path("hand_landmarker.task").exists():
    urllib.request.urlretrieve(MODEL_URL, "hand_landmarker.task")
landmarker = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
))


def image_features(path):
    """126 features for one photo, or None when MediaPipe finds no hand."""
    rgb = np.asarray(Image.open(path).convert("RGB"))
    result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    hands, spare = {"Left": None, "Right": None}, []
    for points, cats in zip(result.hand_landmarks, result.handedness):
        pts = [(p.x, p.y, p.z) for p in points]
        side = cats[0].category_name if cats else None
        if side in hands and hands[side] is None:
            hands[side] = pts
        else:
            spare.append(pts)
    for pts in spare:
        side = "Left" if hands["Left"] is None else "Right" if hands["Right"] is None else None
        if side:
            hands[side] = pts
    if hands["Left"] is None and hands["Right"] is None:
        return None
    return to_features(hands["Left"], hands["Right"], aspect=rgb.shape[1] / rgb.shape[0])


def extract(items, source):
    """items: list of (image_path, letter). Returns Sample dicts (PRD §8) and per-letter stats."""
    samples, tried, found = [], Counter(), Counter()
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    for i, (path, letter) in enumerate(items):
        tried[letter] += 1
        feats = image_features(path)
        if feats is not None:
            found[letter] += 1
            samples.append({"id": str(uuid.uuid4()), "label": letter, "kind": "letters",
                            "frames": [feats], "recordedBy": source, "createdAt": now})
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(items)} gambar diproses")
    total_t, total_f = sum(tried.values()), sum(found.values())
    print(f"{source}: tangan terdeteksi di {total_f}/{total_t} gambar ({100 * total_f / max(total_t, 1):.1f}%)")
    print("  " + " ".join(f"{l}={found[l]}/{tried[l]}" for l in LETTERS if tried[l]))
    return samples

## 4. Dataset utama: Kaggle (Meisya Vira)

Diunduh otomatis lewat `kagglehub`. Jika gagal (mis. butuh login), unduh zip dari halaman Kaggle lalu unggah saat diminta.
Hasil ekstraksi disimpan ke `data/kaggle-letters.json` supaya tidak perlu diulang.

In [ ]:
import zipfile

KAGGLE_CACHE = DATA_DIR / "kaggle-letters.json"
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if KAGGLE_CACHE.exists():
    kaggle = json.loads(KAGGLE_CACHE.read_text())
    print("Memakai cache:", len(kaggle), "sampel")
else:
    try:
        import kagglehub
        kaggle_root = Path(kagglehub.dataset_download("meisyavira/abjad-bahasa-isyarat-indonesia-bisindo"))
    except Exception as err:  # noqa: BLE001 - any download failure falls back to a manual upload
        print("kagglehub gagal:", err)
        print("Unggah zip dataset dari halaman Kaggle.")
        zip_name = next(iter(files.upload())) if IN_COLAB else sorted(glob.glob("../datasets/kaggle-bisindo/*.zip"))[0]
        kaggle_root = Path("kaggle")
        zipfile.ZipFile(zip_name).extractall(kaggle_root)

    # Folder name = letter (e.g. dataset_merged/A/xxx.jpg).
    items = sorted(
        (p, p.parent.name.upper()) for p in kaggle_root.rglob("*")
        if p.suffix.lower() in IMAGE_EXT and p.parent.name.upper() in LETTERS
    )
    print("Gambar ditemukan:", len(items))
    assert items, "Struktur folder tidak dikenali; cek isi kaggle_root."
    kaggle = extract(items, "kaggle")
    KAGGLE_CACHE.write_text(json.dumps(kaggle))

## 5. (Opsional) Roboflow sebagai tambahan data latih

Set `USE_ROBOFLOW = True`, lalu unggah zip ekspor **YOLOv8** dari BISINDO Dataset v3. Kelas dibaca dari `data.yaml`;
kelas `BISINDO-Signs` (bukan huruf) dilewati. Gambarnya sudah grayscale dan ditarik ke 244×244; catat persentase deteksinya.

In [ ]:
USE_ROBOFLOW = False

roboflow = []
if USE_ROBOFLOW:
    import yaml
    RF_CACHE = DATA_DIR / "roboflow-letters.json"
    if RF_CACHE.exists():
        roboflow = json.loads(RF_CACHE.read_text())
    else:
        zip_name = next(iter(files.upload())) if IN_COLAB else sorted(glob.glob("../datasets/roboflow-bisindo/*.zip"))[0]
        rf_root = Path("roboflow")
        zipfile.ZipFile(zip_name).extractall(rf_root)
        names = yaml.safe_load((rf_root / "data.yaml").read_text())["names"]
        items = []
        for image_path in sorted(rf_root.glob("*/images/*")):
            label_path = image_path.parent.parent / "labels" / (image_path.stem + ".txt")
            tokens = label_path.read_text().split() if label_path.exists() else []
            letter = str(names[int(tokens[0])]).upper() if tokens else ""
            if letter in LETTERS:
                items.append((image_path, letter))
        roboflow = extract(items, "roboflow")
        RF_CACHE.write_text(json.dumps(roboflow))
print("Roboflow:", len(roboflow), "sampel")

## 6. Split 70/15/15

Split stratified per huruf dari data Kaggle. Roboflow (jika dipakai) hanya ditambahkan ke data latih.

**Catatan untuk laporan:** foto Kaggle berasal dari sedikit peraga, jadi foto latih dan uji bisa dari orang yang sama.
Akurasi uji di sini cenderung lebih tinggi daripada pada pengguna baru; uji juga langsung di aplikasi dengan orang lain.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
per_label = Counter(s["label"] for s in kaggle)
print("Sampel per huruf:", " ".join(f"{l}={per_label[l]}" for l in LETTERS))
missing = [l for l in LETTERS if per_label[l] < 3]
assert not missing, f"Huruf tanpa cukup sampel (< 3): {missing}"

X_all = np.array([s["frames"][0] for s in kaggle], dtype=np.float32)
y_all = np.array([LETTERS.index(s["label"]) for s in kaggle], dtype=np.int64)
X_train, X_rest, y_train, y_rest = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, random_state=SEED,
    stratify=y_rest if min(Counter(y_rest.tolist()).values()) >= 2 else None)

if roboflow:
    X_rf = np.array([s["frames"][0] for s in roboflow], dtype=np.float32)
    y_rf = np.array([LETTERS.index(s["label"]) for s in roboflow], dtype=np.int64)
    X_train, y_train = np.concatenate([X_train, X_rf]), np.concatenate([y_train, y_rf])
print(f"train={len(X_train)} val={len(X_val)} test={len(X_test)}")

## 7. Augmentasi (PRD §11.2)

Di level landmark, bukan piksel: rotasi ±15° per tangan (bidang x–y, mengelilingi pergelangan), skala ±10%,
noise Gaussian kecil, dan mirror (tukar blok kiri/kanan + balik tanda x). Tangan kosong (semua 0) tetap 0.

In [ ]:
rng = np.random.default_rng(SEED)
AUG_COPIES = 10  # few photos per letter, so augment more

def augment_hand(h):
    if not h.any():
        return h
    pts = h.reshape(LANDMARKS, 3).copy()
    angle = np.deg2rad(rng.uniform(-15, 15))
    c, s = np.cos(angle), np.sin(angle)
    x, y = pts[:, 0].copy(), pts[:, 1].copy()
    pts[:, 0], pts[:, 1] = c * x - s * y, s * x + c * y
    pts *= rng.uniform(0.9, 1.1)
    pts += rng.normal(0, 0.02, pts.shape)
    pts[0] = 0.0  # the wrist stays at the origin, as after normalization
    return pts.reshape(-1)

def mirror(f):
    left, right = f[:PER_HAND].reshape(LANDMARKS, 3).copy(), f[PER_HAND:].reshape(LANDMARKS, 3).copy()
    left[:, 0] *= -1
    right[:, 0] *= -1
    return np.concatenate([right.reshape(-1), left.reshape(-1)])

def augment(X, y):
    xs, ys = [X], [y]
    for _ in range(AUG_COPIES):
        batch = []
        for f in X:
            f = mirror(f) if rng.random() < 0.5 else f
            batch.append(np.concatenate([augment_hand(f[:PER_HAND]), augment_hand(f[PER_HAND:])]))
        xs.append(np.array(batch, dtype=np.float32))
        ys.append(y)
    return np.concatenate(xs), np.concatenate(ys)

X_train_aug, y_train_aug = augment(X_train, y_train)
print("train setelah augmentasi:", X_train_aug.shape)

## 8. Latih model

In [ ]:
import tf_keras as keras

keras.utils.set_random_seed(SEED)
model = keras.Sequential([
    keras.layers.Input(shape=(FEATURES,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(len(LETTERS), activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val, y_val),
    epochs=150, batch_size=32, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
)

## 9. Evaluasi (untuk laporan)

Target PRD: akurasi abjad ≥ 90% pada data uji.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

REPORT_DIR = Path("report")
REPORT_DIR.mkdir(exist_ok=True)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi uji: {acc:.3f} ({len(X_test)} sampel)")

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
present = sorted(set(y_test.tolist()) | set(y_pred.tolist()))
report = classification_report(y_test, y_pred, labels=present,
                               target_names=[LETTERS[i] for i in present], zero_division=0)
print(report)
(REPORT_DIR / "letters_classification_report.txt").write_text(report)

cm = confusion_matrix(y_test, y_pred, labels=list(range(len(LETTERS))))
np.savetxt(REPORT_DIR / "letters_confusion_matrix.csv", cm, fmt="%d", delimiter=",",
           header=",".join(LETTERS), comments="")
fig, ax = plt.subplots(figsize=(11, 11))
ConfusionMatrixDisplay(cm, display_labels=LETTERS).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix abjad (akurasi uji {acc:.1%})")
fig.savefig(REPORT_DIR / "letters_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history["accuracy"], label="train")
ax.plot(history.history["val_accuracy"], label="val")
ax.set_xlabel("epoch"); ax.set_ylabel("akurasi"); ax.legend()
fig.savefig(REPORT_DIR / "letters_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Ekspor ke TF.js

Ditulis langsung ke format *layers-model* TF.js (`model.json` + satu `.bin`) tanpa paket `tensorflowjs`
(paket itu gagal di Python 3.13 / NumPy 2, DECISIONS #23). Format ini dicek oleh `src/lib/tfModel.test.ts`.
Salin isi folder `letters/` ke `public/models/letters/`; aplikasi otomatis keluar dari mode mock.

In [ ]:
import shutil


def save_tfjs_layers_model(model, out_dir):
    """Write a TF.js layers-model (model.json + group1-shard1of1.bin) from a tf_keras model."""
    out_dir = Path(out_dir)
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    specs, chunks = [], []
    for layer in model.layers:
        for w in layer.weights:
            # TF.js looks weights up as "<layer name>/<weight name>", e.g. "dense/kernel".
            name = f"{layer.name}/{w.name.split('/')[-1].split(':')[0]}"
            value = np.asarray(w.numpy(), dtype="<f4")
            specs.append({"name": name, "shape": list(value.shape), "dtype": "float32"})
            chunks.append(value.tobytes())
    (out_dir / "group1-shard1of1.bin").write_bytes(b"".join(chunks))
    (out_dir / "model.json").write_text(json.dumps({
        "format": "layers-model",
        "generatedBy": f"tf_keras {keras.__version__}",
        "convertedBy": "IsyaratKu save_tfjs_layers_model",
        "modelTopology": json.loads(model.to_json()),
        "weightsManifest": [{"paths": ["group1-shard1of1.bin"], "weights": specs}],
    }))
    return specs


OUT_DIR = Path("letters")
specs = save_tfjs_layers_model(model, OUT_DIR)
assert model.output_shape[-1] == len(LETTERS)

# Check: a plain NumPy forward pass using ONLY the exported files must match Keras.
buf = np.frombuffer((OUT_DIR / "group1-shard1of1.bin").read_bytes(), dtype="<f4")
params, offset = {}, 0
for spec in specs:
    n = int(np.prod(spec["shape"]))
    params[spec["name"]] = buf[offset:offset + n].reshape(spec["shape"])
    offset += n
h = X_test
for layer in model.layers:
    if isinstance(layer, keras.layers.Dense):
        h = h @ params[f"{layer.name}/kernel"] + params[f"{layer.name}/bias"]
        if layer.get_config()["activation"] == "relu":
            h = np.maximum(h, 0)
        else:
            h = np.exp(h - h.max(axis=1, keepdims=True))
            h = h / h.sum(axis=1, keepdims=True)
diff = np.abs(h - model.predict(X_test, verbose=0)).max()
assert diff < 1e-4, diff
print(f"Ekspor OK (selisih maks {diff:.1e}):", sorted(p.name for p in OUT_DIR.iterdir()))

if IN_COLAB:
    shutil.make_archive("letters-model", "zip", OUT_DIR)
    shutil.make_archive("letters-report", "zip", REPORT_DIR)
    files.download("letters-model.zip")
    files.download("letters-report.zip")